In [1]:
from tokenizers import Tokenizer, decoders
from transformers import AutoTokenizer
from datasets import load_dataset

In [2]:
import json, pickle, os
from tqdm.notebook import tqdm
import unicodedata
from math import ceil
from collections import Counter

In [3]:
import tokenizer_edge_cases_char as ec

In [4]:
model_dir = "/Users/nirajh/Code/Language_adaptation/olmo2-7b-stage1"

#### Loading the tokenizer and modifying its regex pattern

In [5]:
# Loading the tokenizer
tok = AutoTokenizer.from_pretrained(model_dir)

In [6]:
## Changing the regex pattern in the pre tokenizer, so that kannada words are split correctly
tj = json.loads(tok.backend_tokenizer.to_str())        # Getting the tokenizer's vocab, merges, special tokens, and pre-processing configs
split = tj["pre_tokenizer"]["pretokenizers"][0]        # Fetches the first pre-tokenizer (they are usually chained together under a Sequence)
old = split["pattern"]["Regex"]
new = old.replace(r"[^\r\n\p{L}\p{N}]?\p{L}+", r"[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+")
assert new != old, "pattern not found; print(old) and adjust"
split["pattern"]["Regex"] = new

In [7]:
# Checking its output
kn_tok = Tokenizer.from_str(json.dumps(tj))
text = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
print([text[s:e] for _, (s, e) in kn_tok.pre_tokenizer.pre_tokenize_str(text)])

['ನಮಸ್ಕಾರ', ' ಕನ್ನಡ', ' ಭಾಷೆಯಲ್ಲಿ']


#### Loading the eval set

In [8]:
kn_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/kn_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        kn_texts.append(data["text"])

print(f"Successfully read {len(kn_texts)} lines.")   # Should be 1997


en_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/en_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        en_texts.append(data["text"])

print(f"Successfully read {len(en_texts)} lines.")   # Should be 2000 lines

Successfully read 1997 lines.
Successfully read 2000 lines.


#### Loading the new Unigram Tokens

In [9]:
# Loading the unigram tokenizer
u_tok = Tokenizer.from_file("/Users/nirajh/Code/Language_adaptation/tokenizers/unigram/unigram_4000_char.json")

# Getting the original vocabulary of the OLMo tokenizer
orig_vocab = tok.get_vocab()

In [10]:
# Fetching the tokens that are in the Unigram tokenizer's vocab, but are not present in OLMo's original tokenizer
u_new_tokens = ec.unigram_char_new_tokens(u_tok, orig_vocab)
print("Unigram new Kannada tokens:", len(u_new_tokens))

Unigram new Kannada tokens: 3998


In [11]:
print([(t.value, t.id) for t in u_tok.model.tokenize("ಕa")], u_tok.token_to_id("<unk>"))

[('ಕ', 14), ('a', 0)] 0


In [12]:
bpe_model = kn_tok.model
bytelevel = decoders.ByteLevel()

In [ ]:
seg = ec.hybrid_char_segmenter(u_tok, kn_tok.model)         # Uses BPE for UNK tokens and the Unigram tokenizer for known tokens
pre = kn_tok.pre_tokenizer

def hybrid_stats(seg, texts, lang):
    n_tok = n_words = n_chars = n_frag = 0
    for t in tqdm(texts, leave=False, desc=f"Tokenizer stats on {lang} eval set"):
        pieces = [p for w in ec.pretokens(t, pre) for p in seg(w)]              # pretokens() normalizes and then runs the pre-tokenizer's two steps
        # The text is pre-tokenized to w, and the hybrid_char_segmenter (seg) returns the tokens from the Unigram tok and BPE toke (fallback) for unknown tokens
        n_tok += len(pieces)
        n_words += len(t.split())
        n_chars += len(t)
        n_frag += sum("\ufffd" in ec.to_text(p) for p in pieces)
    return {"fertility (tokens/word)": n_tok / n_words,
            "chars/token": n_chars / n_tok,
            "fragment rate": n_frag / n_tok}

#### Calculating stats for unigram model with vocab size 4000

In [15]:
out_dir = "/Users/nirajh/Code/Language_adaptation/tokenizers/unigram"

In [16]:
# Counting the unigram tokens to find the most frequent ones (in the eval set)
uni_counts = Counter()

for t in kn_texts:
    for w in ec.pretokens(t, pre):
        if ec.is_kannada(w):
            uni_counts.update(seg(w))

In [17]:
# Viewing the 25 most common unigram tokens
total = sum(uni_counts.values())

for p, c in uni_counts.most_common(25):
    print(f"{bytelevel.decode([p])!r:20} {c:>8,}  ({c / total:.2%})")

' '                    23,906  (2.51%)
'ದ'                    17,989  (1.89%)
'ು'                    16,409  (1.72%)
'ರ'                    10,364  (1.09%)
'ಯ'                     9,891  (1.04%)
'ಿ'                     8,756  (0.92%)
'ನ'                     8,458  (0.89%)
'ವ'                     6,495  (0.68%)
'ಗೆ'                    5,887  (0.62%)
'ೆ'                     5,703  (0.60%)
' ಈ'                    5,523  (0.58%)
'ಸ'                     5,407  (0.57%)
'ಗಳು'                   5,160  (0.54%)
' ಮತ್ತು'                5,085  (0.53%)
'ರು'                    4,934  (0.52%)
'ಗಳ'                    4,872  (0.51%)
'್'                     4,829  (0.51%)
'ಕ'                     4,802  (0.50%)
'ದಲ್ಲಿ'                 4,782  (0.50%)
'ಾ'                     4,285  (0.45%)
'ಕ್ಕೆ'                  4,219  (0.44%)
' ಅ'                    4,219  (0.44%)
'ಮ'                     4,203  (0.44%)
'ವನ್ನು'                 4,192  (0.44%)
'ಗಳನ್ನು'                4,168  (0.44%)


The most frequent tokens are individual Kannada characters and some commonly occurring suffixes and words. There are no split characters in this list.

In [19]:
seg = ec.hybrid_char_segmenter(u_tok, kn_tok.model)

kn_stats_4k = hybrid_stats(seg, kn_texts, "Kannada")
en_stats_4k = hybrid_stats(seg, en_texts, "English")

Tokenizer stats on Kannada eval set:   0%|          | 0/1997 [00:00<?, ?it/s]

Tokenizer stats on English eval set:   0%|          | 0/2000 [00:00<?, ?it/s]

In [20]:
print("Stats on Kannada eval set: \n",*kn_stats_4k.items(), sep='\n')
print("\n\nStats on English eval set: \n",*en_stats_4k.items(), sep='\n')

Stats on Kannada eval set: 

('fertility (tokens/word)', 2.3129565259601157)
('chars/token', 3.4951511590135156)
('fragment rate', 0.008757842639871817)


Stats on English eval set: 

('fertility (tokens/word)', 1.2804896737381426)
('chars/token', 4.7985698471112705)
('fragment rate', 0.0013623353179945945)
